# Crear la tabla de usuarios

## Qué hace

Crea la tabla de usuarios si no existe. No guarda usuarios.

## Qué pasa en la base de datos

Si la tabla no estaba, pasa a estar. Si ya estaba, sigue igual. No se insertan filas.

## Qué acción usa

No usa logic. Usa `createUsersTable` y lista los nombres de las tablas, igual que el cliente de terminal.

## Anotaciones

No borra filas. Se abre con `pnpm jupyter` desde la app.


In [1]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent
if not (app / "logic").is_dir():
    raise SystemExit("No encuentro la app.")

code = r"""
import Table from "cli-table3";
import { createUsersTable } from "./queries/users.js";
import { db } from "./db/connection.js";

createUsersTable();

const result = db
  .prepare("SELECT name FROM sqlite_master WHERE type = 'table'")
  .all();

const table = new Table({
  head: ["name"],
  style: { head: ["cyan", "bold"] },
});

for (const row of result) {
  table.push([row.name]);
}

console.log(table.toString());

if (result.length === 0) {
  console.log("No hay filas.");
}
"""

env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

completed = subprocess.run(
    ["node", "--input-type=module", "-e", code],
    cwd=app,
    capture_output=True,
    text=True,
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")


┌───────┐
│ name  │
├───────┤
│ users │
└───────┘
